# Sobolev-trained data-driven baseline — advisor follow-up review, point 4

The existing data-driven baseline (`train_data_driven.py`) trains on plain relative-L2 displacement error. This adds a **Sobolev / H1-type loss**: (1-w)\*rel_L2(u) + w\*rel_L2(grad_u), matching both the displacement field AND its gradient (the deformation gradient F=I+du/dX) to the FEM reference — the same H1 semi-norm this study already reports as a QoI, now also used as a training signal.

* Needs the SAME dataset the physics-informed and plain data-driven runs already used (no new FEM solves).
* GPU strongly preferred.
* `--sobolev_grad_weight` defaults to 0.5; change it below to try a different balance.
* Compare the resulting `best_val_rel_L2` against both the plain data-driven baseline (`--loss rel_l2`) and the physics-informed result for the same case.

In [ ]:
# =====================================================================
#  CELL — Sobolev-trained data-driven operator (advisor follow-up, point 4)
#  GPU strongly preferred. Needs NO new FEM solves: the labels and their
#  implied deformation gradient are both derived from the same .npz the
#  physics-informed and plain data-driven runs already used.
# =====================================================================
import os, subprocess, sys

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
                    'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE - this will be slow, prefer Runtime > Change runtime type > GPU')

R = '/content/drive/MyDrive/pfem_run'
OUT = f'{R}/sobolev_dd'
os.makedirs(OUT, exist_ok=True)

# --- the case, and the settings that MUST match the other DD/PI runs ---
GEOMETRY  = 'B1'
MATERIAL  = 'neo_hookean'
DATA      = f'{R}/results/datasets/B1_neo_hookean/hyperelastic_training_data_q4.npz'
OPT_STEPS = 75000    # matched to the physics-informed/plain-DD run's own step count
BATCH     = 8
GRAD_WEIGHT = 0.5    # weight on the deformation-gradient (H1) term
# --------------------------------------------------------------------------

assert os.path.exists(DATA), (
    f'dataset not found: {DATA}\n'
    'Check the path -- it must be the SAME .npz the physics-informed/plain-'
    'data-driven models trained on, or the comparison is not like-for-like.')
print('dataset:', DATA, f'({os.path.getsize(DATA)/1e6:.1f} MB)')

run([
    sys.executable, '-u', '-m', 'omar_pfem.train_data_driven',
    '--geometry', GEOMETRY, '--material', MATERIAL,
    '--path', DATA,
    '--ntrain', '800', '--ntest', '200',
    '--batch_size', str(BATCH),
    '--opt_steps', str(OPT_STEPS),
    '--loss', 'sobolev',
    '--sobolev_grad_weight', str(GRAD_WEIGHT),
    '--eval_every', '2000',
    '--out_dir', f'{OUT}/{GEOMETRY}_{MATERIAL}',
])

print(f"\nDone. Result: {OUT}/{GEOMETRY}_{MATERIAL}/data_driven_{GEOMETRY}_{MATERIAL}.json")
print('Compare best_val_rel_L2 against the plain data-driven baseline (--loss rel_l2,')
print('Round6_Data_Driven.ipynb) and the physics-informed result for the same case.')
print('history.json also logs val_term/grad_term separately at every eval step.')